# 01 - Sentiment Dataset Audit

This notebook validates the raw student-feedback sentiment data before preprocessing. It is read-only: the cleaned data is created in notebook 02.

In [ ]:
from pathlib import Path

import pandas as pd

RANDOM_STATE = 42
AI_DIR = Path.cwd().resolve().parent
RAW_DATA_PATH = AI_DIR / 'datasets' / 'student_feedback_sentiment_dataset.csv'

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f'Raw dataset not found: {RAW_DATA_PATH}')

df = pd.read_csv(RAW_DATA_PATH)
df.head()

In [ ]:
required_columns = {'feedback_text', 'sentiment'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

print(f'Rows: {len(df):,}')
print(f'Columns: {df.columns.tolist()}')
print('\nMissing values:')
display(df[list(required_columns)].isna().sum().rename('missing').to_frame())
print(f'Exact duplicate rows: {df.duplicated().sum():,}')
print(f'Blank feedback rows: {df["feedback_text"].astype("string").str.strip().eq("").sum():,}')

In [ ]:
label_counts = df['sentiment'].value_counts().rename_axis('sentiment').to_frame('count')
label_counts['percentage'] = (label_counts['count'] / len(df) * 100).round(2)
display(label_counts)

ax = label_counts['count'].sort_index().plot.bar(
    title='Raw sentiment-label distribution', ylabel='Rows', rot=0, color=['#c96a54', '#7588a8', '#5b9b78']
)
ax.bar_label(ax.containers[0])

In [ ]:
text_lengths = df['feedback_text'].astype('string').str.len()
word_counts = df['feedback_text'].astype('string').str.split().str.len()

display(pd.DataFrame({'characters': text_lengths.describe(), 'words': word_counts.describe()}))

for label in sorted(df['sentiment'].unique()):
    print(f'\n{label.upper()} examples')
    display(df.loc[df['sentiment'].eq(label), ['feedback_text', 'sentiment']].sample(3, random_state=RANDOM_STATE))

## Audit decision

The training contract is a non-empty `feedback_text` paired with one of `negative`, `neutral`, or `positive`. Notebook 02 normalizes text and labels, removes unusable rows and duplicates, then writes deterministic stratified train, validation, and test splits.